# Understanding the Data

This notebook performs an initial audit of the Airbnb listings dataset. The goal is to confirm the dataset loads correctly, inspect its structure, and document basic data quality signals before any exploratory analysis, preprocessing, or modeling decisions are made.

## Setup

The setup keeps the loading step explicit and easy to follow by using `pandas.read_csv(...)` directly.

In [18]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## Load the dataset

This notebook loads the CSV directly from the project data folder using a simple relative path from the `notebooks/` directory.

In [ ]:
DATA_PATH = Path("../data/listings.csv")

print("Working directory:", Path.cwd())
print(f"Dataset path: {DATA_PATH}")
listings = pd.read_csv(DATA_PATH)

print(f"Shape: {listings.shape[0]:,} rows x {listings.shape[1]} columns")
listings.head()

### Interpretation

The dataset now loads through a direct CSV read from a relative notebook path, which keeps the setup simple and easy to understand. Printing the working directory makes it obvious why the file path uses `../data/listings.csv`. 

### Interpretation

The dataset now loads through a direct CSV read, which keeps the notebook setup simple and easy to understand. The printed working directory and explicit `DATA_PATH` make it clear where the file is being loaded from.

In [4]:
columns_df = pd.DataFrame({"column": listings.columns})
display(columns_df)

,column
0,id
1,name
2,host_id
3,host_name
4,neighbourhood_group
5,neighbourhood
6,latitude
7,longitude
8,room_type
9,price


In [5]:
target_present = "price" in listings.columns

if target_present:
    interpretation = (
        "**Interpretation:** The dataset includes the target column `price`, so it is structurally suitable "
        "for a supervised price prediction workflow. The remaining columns span identifiers, host details, "
        "location, availability, and review activity."
    )
else:
    interpretation = (
        "**Interpretation:** The target column `price` is missing. That would block the planned modeling work "
        "and would need to be resolved before proceeding."
    )

display(Markdown(interpretation))

**Interpretation:** The dataset includes the target column `price`, so it is structurally suitable for a supervised price prediction workflow. The remaining columns span identifiers, host details, location, availability, and review activity.

## Data types

Inspecting dtypes early helps identify which fields are already numeric and which ones may require later parsing or encoding.

In [6]:
dtype_summary = (
    listings.dtypes.astype(str)
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(dtype_summary)

,column,dtype
0,id,int64
1,name,str
2,host_id,int64
3,host_name,str
4,neighbourhood_group,str
5,neighbourhood,str
6,latitude,float64
7,longitude,float64
8,room_type,str
9,price,float64


In [7]:
numeric_columns = listings.select_dtypes(include=["number"]).shape[1]
non_numeric_columns = listings.shape[1] - numeric_columns

display(Markdown(
    f"**Interpretation:** The raw dataset currently contains **{numeric_columns} numeric columns** and "
    f"**{non_numeric_columns} non-numeric columns**. This mix is typical for listings data and suggests that "
    "future preprocessing will need to handle both structured numeric fields and text or date-like fields."
))

**Interpretation:** The raw dataset currently contains **11 numeric columns** and **7 non-numeric columns**. This mix is typical for listings data and suggests that future preprocessing will need to handle both structured numeric fields and text or date-like fields.

## Missing values

Missing-value checks help identify which columns may need later treatment, documentation, or exclusion decisions.

In [8]:
missing_summary = pd.DataFrame({
    "column": listings.columns,
    "missing_count": listings.isna().sum().values,
})
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(listings)).round(4)
missing_summary = missing_summary.sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)
display(missing_summary)

,column,missing_count,missing_pct
0,license,31079,0.8549
1,price,14938,0.4109
2,last_review,11346,0.3121
3,reviews_per_month,11346,0.3121
4,host_name,1457,0.0401
5,name,2,0.0001
6,availability_365,0,0.0000
7,calculated_host_listings_count,0,0.0000
8,host_id,0,0.0000
9,id,0,0.0000


In [9]:
columns_with_missing = missing_summary.loc[missing_summary["missing_count"] > 0]

if columns_with_missing.empty:
    interpretation = (
        "**Interpretation:** No missing values were detected in the current raw file, so completeness does not appear "
        "to be an immediate issue at the audit stage."
    )
else:
    top_missing = columns_with_missing.iloc[0]
    interpretation = (
        f"**Interpretation:** Missingness is present in **{len(columns_with_missing)} columns**. "
        f"The most affected field is **`{top_missing['column']}`**, with **{int(top_missing['missing_count']):,}** "
        f"missing values (**{top_missing['missing_pct']:.2%}** of rows). We are only documenting this now; "
        "treatment decisions belong in a later preprocessing step."
    )

display(Markdown(interpretation))

**Interpretation:** Missingness is present in **6 columns**. The most affected field is **`license`**, with **31,079** missing values (**85.49%** of rows). We are only documenting this now; treatment decisions belong in a later preprocessing step.

## Duplicate rows

A duplicate-row check helps confirm whether the raw file appears to contain repeated records before we make any cleaning decisions.

In [11]:
duplicate_row_count = int(listings.duplicated().sum())
duplicate_row_pct = duplicate_row_count / len(listings)

print(f"Duplicate rows: {duplicate_row_count}")
print(f"Duplicate row percentage: {duplicate_row_pct:.2%}")

Duplicate rows: 0
Duplicate row percentage: 0.00%


In [12]:
if duplicate_row_count == 0:
    interpretation = (
        "**Interpretation:** No fully duplicated rows were detected, so exact record duplication does not appear to be "
        "a current issue in the raw data."
    )
else:
    interpretation = (
        f"**Interpretation:** The raw file contains **{duplicate_row_count:,} fully duplicated rows** "
        f"(**{duplicate_row_pct:.2%}** of the dataset). Whether those rows should be removed will be decided later, "
        "during preprocessing."
    )

display(Markdown(interpretation))

**Interpretation:** No fully duplicated rows were detected, so exact record duplication does not appear to be a current issue in the raw data.

## Target column: `price`

The target review in this notebook is limited to a quick preview and a type check. Distributional analysis and outlier treatment belong in later steps.

In [13]:
display(listings[["price"]].head())
display(listings["price"].describe().to_frame(name="price"))

,price
0,87.0
1,80.0
2,99.0
3,312.0
4,111.0


,price
count,21415.000000
mean,519.622881
std,3658.430988
min,9.000000
25%,90.000000
50%,154.000000
75%,269.000000
max,50138.000000


In [14]:
price_dtype = listings["price"].dtype
price_is_numeric = pd.api.types.is_numeric_dtype(listings["price"])
coerced_price = pd.to_numeric(listings["price"], errors="coerce")
new_nulls_from_coercion = int(coerced_price.isna().sum() - listings["price"].isna().sum())

print(f"price dtype: {price_dtype}")
print(f"Is numeric: {price_is_numeric}")
print(f"Additional nulls introduced by numeric coercion: {new_nulls_from_coercion}")

price dtype: float64
Is numeric: True
Additional nulls introduced by numeric coercion: 0


In [15]:
if price_is_numeric and new_nulls_from_coercion == 0:
    interpretation = (
        "**Interpretation:** `price` is already stored as a numeric field in the raw dataset, so no immediate string-to-number "
        "cleaning is required. Later steps may still examine skew, outliers, or transformations, but those decisions are "
        "outside the scope of this audit notebook."
    )
else:
    interpretation = (
        "**Interpretation:** `price` is not cleanly numeric yet, or coercion introduces additional missing values. "
        "That means the target will need explicit cleaning before any modeling work begins."
    )

display(Markdown(interpretation))

**Interpretation:** `price` is already stored as a numeric field in the raw dataset, so no immediate string-to-number cleaning is required. Later steps may still examine skew, outliers, or transformations, but those decisions are outside the scope of this audit notebook.

## Audit takeaway

This notebook establishes the basic structure and data-quality baseline of the raw Airbnb listings file. The next step should turn these audit observations into explicit preprocessing and EDA decisions, without mixing that work into this initial inspection notebook.